## Transform Orders Data - String to JSON
1. Pre-process the JSON String to fix the Data Quality Issues
1. Transform JSON String to JSON Object
1. Write transformed data to the silver schema

In [0]:
SELECT * 
  FROM gizmobox.bronze.v_orders;

value
"{""order_id"": 1, ""customer_id"": 6973, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 10:13:59"", ""total_amount"": 499, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 8, ""name"": ""Gaming Console"", ""category"": ""Electronics"", ""price"": 499, ""quantity"": 1, ""details"": {""brand"": ""Sony"", ""color"": ""Blue""}}], ""order_status"": ""Completed""}"
"{""order_id"": 2, ""customer_id"": 3532, ""order_date"": 2025-01-19, ""transaction_timestamp"": ""2025-01-19 00:05:13"", ""total_amount"": 985, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 4, ""name"": ""Smartwatch"", ""category"": ""Electronics"", ""price"": 299, ""quantity"": 2, ""details"": {""brand"": ""Canon"", ""color"": ""Gray""}}, {""item_id"": 7, ""name"": ""External Hard Drive"", ""category"": ""Electronics"", ""price"": 129, ""quantity"": 3, ""details"": {""brand"": ""Dell"", ""color"": ""Blue""}}], ""order_status"": ""Cancelled""}"
"{""order_id"": 3, ""customer_id"": 3532, ""order_date"": ""2025-01-08"", ""transaction_timestamp"": ""2025-01-08 23:11:00"", ""total_amount"": 597, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 3, ""name"": ""Wireless Headphones"", ""category"": ""Electronics"", ""price"": 199, ""quantity"": 3, ""details"": {""brand"": ""Apple"", ""color"": ""White""}}], ""order_status"": ""Completed""}"
"{""order_id"": 4, ""customer_id"": 3532, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 05:49:26"", ""total_amount"": 999, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 2, ""name"": ""Laptop"", ""category"": ""Electronics"", ""price"": 999, ""quantity"": 1, ""details"": {""brand"": ""Microsoft"", ""color"": ""Black""}}], ""order_status"": ""Cancelled""}"
"{""order_id"": 8, ""customer_id"": 9179, ""order_date"": ""2025-01-02"", ""transaction_timestamp"": ""2025-01-02 13:11:15"", ""total_amount"": 399, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 5, ""name"": ""Tablet"", ""category"": ""Electronics"", ""price"": 399, ""quantity"": 1, ""details"": {""brand"": ""LG"", ""color"": ""White""}}], ""order_status"": ""Completed""}"
"{""order_id"": 14, ""customer_id"": 7207, ""order_date"": ""2025-01-01"", ""transaction_timestamp"": ""2025-01-01 18:22:26"", ""total_amount"": 597, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 3, ""name"": ""Wireless Headphones"", ""category"": ""Electronics"", ""price"": 199, ""quantity"": 3, ""details"": {""brand"": ""Bose"", ""color"": ""Blue""}}], ""order_status"": ""Cancelled""}"
"{""order_id"": 18, ""customer_id"": 4468, ""order_date"": ""2025-01-22"", ""transaction_timestamp"": ""2025-01-22 21:12:08"", ""total_amount"": 799, ""payment_method"": ""Credit Card"", ""items"": [{""item_id"": 10, ""name"": ""Drone"", ""category"": ""Electronics"", ""price"": 799, ""quantity"": 1, ""details"": {""brand"": ""Dell"", ""color"": ""Black""}}], ""order_status"": ""Pending""}"
"{""order_id"": 22, ""customer_id"": 4761, ""order_date"": 2025-01-01, ""transaction_timestamp"": ""2025-01-01 15:49:19"", ""total_amount"": 199, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 3, ""name"": ""Wireless Headphones"", ""category"": ""Electronics"", ""price"": 199, ""quantity"": 1, ""details"": {""brand"": ""HP"", ""color"": ""Silver""}}], ""order_status"": ""Pending""}"
"{""order_id"": 26, ""customer_id"": 7007, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 19:01:25"", ""total_amount"": 699, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 1, ""name"": ""Smartphone"", ""category"": ""Electronics"", ""price"": 699, ""quantity"": 1, ""details"": {""brand"": ""Samsung"", ""color"": ""Silver""}}], ""order_status"": ""Shipped""}"
"{""order_id"": 29, ""customer_id"": 5953, ""order_date"": ""2025-01-18"", ""transaction_timestamp"": ""2025-01-18 03:05:04"", ""total_amount"": 129, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 7, ""name"":

### 1. Pre-process the JSON String to fix the Data Quality Issues
[regexp_replace function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace)

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_fixed
AS
SELECT value,
       regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value 
  FROM gizmobox.bronze.v_orders;

### 2. Transform JSON String to JSON Object
- Function [schema_of_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)
- Function [from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)

In [0]:
SELECT schema_of_json(fixed_value) AS schema,
       fixed_value
  FROM tv_orders_fixed
 LIMIT 1;

schema,fixed_value
"STRUCT, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>","{""order_id"": 1, ""customer_id"": 6973, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 10:13:59"", ""total_amount"": 499, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 8, ""name"": ""Gaming Console"", ""category"": ""Electronics"", ""price"": 499, ""quantity"": 1, ""details"": {""brand"": ""Sony"", ""color"": ""Blue""}}], ""order_status"": ""Completed""}"


In [0]:
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value,
       fixed_value
  FROM tv_orders_fixed;

json_value,fixed_value
"List(6973, List(List(Electronics, List(Sony, Blue), 8, Gaming Console, 499, 1)), 2025-01-05, 1, Completed, Bank Transfer, 499, 2025-01-05 10:13:59)","{""order_id"": 1, ""customer_id"": 6973, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 10:13:59"", ""total_amount"": 499, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 8, ""name"": ""Gaming Console"", ""category"": ""Electronics"", ""price"": 499, ""quantity"": 1, ""details"": {""brand"": ""Sony"", ""color"": ""Blue""}}], ""order_status"": ""Completed""}"


### 3. Write transformed data to the silver schema

In [0]:
DROP TABLE IF EXISTS gizmobox.silver.orders_json;
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders_json
AS
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value
  FROM tv_orders_fixed;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM gizmobox.silver.orders_json;

json_value
"List(6973, List(List(Electronics, List(Sony, Blue), 8, Gaming Console, 499, 1)), 2025-01-05, 1, Completed, Bank Transfer, 499, 2025-01-05 10:13:59)"
"List(3532, List(List(Electronics, List(Canon, Gray), 4, Smartwatch, 299, 2), List(Electronics, List(Dell, Blue), 7, External Hard Drive, 129, 3)), 2025-01-19, 2, Cancelled, PayPal, 985, 2025-01-19 00:05:13)"
"List(3532, List(List(Electronics, List(Apple, White), 3, Wireless Headphones, 199, 3)), 2025-01-08, 3, Completed, Bank Transfer, 597, 2025-01-08 23:11:00)"
"List(3532, List(List(Electronics, List(Microsoft, Black), 2, Laptop, 999, 1)), 2025-01-05, 4, Cancelled, Bank Transfer, 999, 2025-01-05 05:49:26)"
"List(9179, List(List(Electronics, List(LG, White), 5, Tablet, 399, 1)), 2025-01-02, 8, Completed, PayPal, 399, 2025-01-02 13:11:15)"
"List(7207, List(List(Electronics, List(Bose, Blue), 3, Wireless Headphones, 199, 3)), 2025-01-01, 14, Cancelled, PayPal, 597, 2025-01-01 18:22:26)"
"List(4468, List(List(Electronics, List(Dell, Black), 10, Drone, 799, 1)), 2025-01-22, 18, Pending, Credit Card, 799, 2025-01-22 21:12:08)"
"List(4761, List(List(Electronics, List(HP, Silver), 3, Wireless Headphones, 199, 1)), 2025-01-01, 22, Pending, PayPal, 199, 2025-01-01 15:49:19)"
"List(7007, List(List(Electronics, List(Samsung, Silver), 1, Smartphone, 699, 1)), 2025-01-05, 26, Shipped, Bank Transfer, 699, 2025-01-05 19:01:25)"
"List(5953, List(List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1), List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1)), 2025-01-18, 29, Pending, Bank Transfer, 129, 2025-01-18 03:05:04)"
